In [1]:
# Instalar dependencias en el kernel actual de Jupyter.
%pip install -q pandas numpy torch transformers scikit-learn==1.7.2 joblib peft datasets accelerate evaluate spacy
%pip install -q https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# Pre-entrega 4: Clasificacion eficiente con LoRA

Este notebook entrena un clasificador Transformer con adaptadores LoRA sobre el mismo corpus AG News utilizado en el Modulo 3. El baseline TF-IDF + Logistic Regression se conserva para la comparacion final.

El entrenamiento requiere una GPU CUDA. Antes de comenzar se verifica `torch.cuda.is_available()`.

## 1. Configuracion y dependencias

In [2]:
from pathlib import Path
import html
import json
import re
import time
import numpy as np
import pandas as pd
import spacy
import torch
from datasets import ClassLabel, Dataset, DatasetDict
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, TaskType, get_peft_model

SEED = 42
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Advertencia: no se detecto una GPU CUDA; el entrenamiento en CPU puede ser lento.")
print("Dispositivo:", "cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)

GPU: NVIDIA GeForce GTX 1650 Ti
Dispositivo: cuda
PyTorch: 2.5.1


## 2. Carga del corpus y conversion de etiquetas

Se conserva el mismo train/test del Modulo 3. El test no se utiliza para entrenar ni para seleccionar hiperparametros.

In [3]:
ROOT = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / "data" / "ag_news_train.csv").exists()),
    None,
 )
if ROOT is None:
    raise FileNotFoundError("No se encontro la carpeta data con AG News.")

train_df = pd.read_csv(ROOT / "data" / "ag_news_train.csv")
test_df = pd.read_csv(ROOT / "data" / "ag_news_test.csv")
labels = sorted(train_df["label"].unique())
label2id = {label: index for index, label in enumerate(labels)}
id2label = {index: label for label, index in label2id.items()}

train_df = train_df.assign(labels=train_df["label"].map(label2id))
test_df = test_df.assign(labels=test_df["label"].map(label2id))

train_dataset = Dataset.from_pandas(train_df[["text", "labels"]], preserve_index=False)
test_dataset = Dataset.from_pandas(test_df[["text", "labels"]], preserve_index=False)
label_feature = ClassLabel(names=labels)
train_dataset = train_dataset.cast_column("labels", label_feature)
test_dataset = test_dataset.cast_column("labels", label_feature)
split = train_dataset.train_test_split(test_size=0.2, seed=SEED, stratify_by_column="labels")
dataset = DatasetDict({"train": split["train"], "validation": split["test"], "test": test_dataset})

print(dataset)
print("Clases:", label2id)

Casting the dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 6400
    })
    validation: Dataset({
        features: ['text', 'labels'],
        num_rows: 1600
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 2000
    })
})
Clases: {'Business': 0, 'Sci_Tech': 1, 'Sports': 2, 'World': 3}


## 3. Tokenizacion y modelo base

**Justificacion del modelo base:** se elige `distilbert-base-uncased` porque el corpus AG News esta en ingles (no requiere un modelo multilingue) y en minusculas normalizadas por el preprocesamiento previo. DistilBERT retiene aproximadamente el 97% del rendimiento de BERT-base con un 40% menos de parametros y hasta 60% mas de velocidad de inferencia, lo cual es clave dado el hardware limitado (GPU unica GTX 1650 Ti) disponible para este checkpoint. Modelos mas grandes como `bert-base-uncased` o `roberta-base` habrian aumentado el costo de entrenamiento sin una mejora justificada para una tarea de clasificacion de 4 clases relativamente simple.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

tokenized_dataset = dataset.map(tokenize_batch, batched=True, remove_columns=["text"])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

Map:   0%|          | 0/6400 [00:00<?, ? examples/s]

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 4. Configuracion LoRA y analisis de parametros

**Justificacion de los hiperparametros LoRA:** la clasificacion de topicos de AG News (4 clases con vocabulario bien diferenciado) es una tarea de complejidad baja-media, por lo que un rango `r=8` alcanza para capturar los patrones especificos de la tarea sin sobreajustar ni añadir parametros innecesarios; un rango mayor (16/32) incrementaria el costo sin evidencia de que la tarea lo requiera. `lora_alpha=16` (razon alpha/r = 2) escala las actualizaciones lo suficiente para que el adaptador tenga impacto real durante el entrenamiento, manteniendo la estabilidad numerica. `lora_dropout=0.1` regulariza el adaptador para evitar sobreajuste dado el tamaño moderado del dataset (8.000 ejemplos). Los modulos objetivo `q_lin` y `v_lin` corresponden a las proyecciones de consulta y valor de la atencion, el punto estandar de insercion de adaptadores LoRA recomendado en el paper original, ya que concentran la mayor parte de la capacidad discriminativa del mecanismo de atencion con el menor numero de parametros entrenables.

In [5]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],
    bias="none",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

total_parameters = sum(parameter.numel() for parameter in model.parameters())
trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
trainable_percentage = 100 * trainable_parameters / total_parameters
print({
    "total_parameters": total_parameters,
    "trainable_parameters": trainable_parameters,
    "trainable_percentage": trainable_percentage,
})

trainable params: 741,124 || all params: 67,697,672 || trainable%: 1.0948
{'total_parameters': 67697672, 'trainable_parameters': 741124, 'trainable_percentage': 1.0947555183286066}


## 5. Entrenamiento y evaluacion

In [6]:
def compute_metrics(eval_prediction):
    logits, labels_array = eval_prediction
    predictions_array = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels_array, predictions_array, average="macro", zero_division=0
    )
    return {
        "accuracy": accuracy_score(labels_array, predictions_array),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
    }

training_args = TrainingArguments(
    output_dir="./resultados_lora",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_strategy="epoch",
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

start_time = time.perf_counter()
train_result = trainer.train()
training_seconds = time.perf_counter() - start_time
validation_metrics = trainer.evaluate(tokenized_dataset["validation"])
print("Tiempo de entrenamiento (segundos):", round(training_seconds, 2))
print("Metricas de validacion:", validation_metrics)

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro
1,0.864194,0.421408,0.869375,0.868601,0.869375,0.868806
2,0.359230,0.357748,0.878125,0.878073,0.878125,0.877627
3,0.331626,0.352003,0.877500,0.877223,0.877500,0.877112


Training Loss,Validation Loss,Epoch,Accuracy,Precision Macro,Recall Macro,F1 Macro
0.331626,0.357748,3,0.878125,0.878073,0.878125,0.877627


Tiempo de entrenamiento (segundos): 270.28
Metricas de validacion: {'eval_loss': 0.3577481806278229, 'eval_accuracy': 0.878125, 'eval_precision_macro': 0.8780731235462788, 'eval_recall_macro': 0.8781249999999999, 'eval_f1_macro': 0.8776266120296758}


## 6. Benchmark final contra TF-IDF

In [7]:
test_output = trainer.predict(tokenized_dataset["test"])
lora_predictions = np.argmax(test_output.predictions, axis=-1)
lora_metrics = compute_metrics((test_output.predictions, np.array(test_dataset["labels"])))
lora_report = classification_report(
    test_dataset["labels"],
    lora_predictions,
    target_names=labels,
    zero_division=0,
)
lora_confusion = confusion_matrix(test_dataset["labels"], lora_predictions)
print(lora_report)
print("Matriz de confusion LoRA:")
print(lora_confusion)

baseline_path = ROOT / "modelo_tfidf_ag_news.joblib"
baseline = __import__("joblib").load(baseline_path)
nlp = spacy.load("en_core_web_sm")
html_tag_re = re.compile(r"<[^>]+>")
url_re = re.compile(r"https?://\S+|www\.\S+")
non_alpha_re = re.compile(r"[^a-zA-Z\s]")
multi_space_re = re.compile(r"\s+")

def baseline_preprocess(text: str) -> str:
    text = html.unescape(text)
    text = html_tag_re.sub(" ", text)
    text = url_re.sub(" ", text)
    text = non_alpha_re.sub(" ", text.lower())
    text = multi_space_re.sub(" ", text).strip()
    doc = nlp(text)
    return " ".join(
        token.lemma_
        for token in doc
        if not token.is_space and not token.is_punct and len(token.lemma_) > 1
    )

test_processed = test_df["text"].fillna("").map(baseline_preprocess)
baseline_predictions = baseline.predict(test_processed)
baseline_precision, baseline_recall, baseline_f1, _ = precision_recall_fscore_support(
    test_df["label"], baseline_predictions, average="macro", zero_division=0
)
baseline_report = classification_report(
    test_df["label"], baseline_predictions, zero_division=0,
)
baseline_confusion = confusion_matrix(test_df["label"], baseline_predictions, labels=labels)
comparison = pd.DataFrame([
    {"modelo": "TF-IDF + LogisticRegression", "precision_macro": baseline_precision, "recall_macro": baseline_recall, "f1_macro": baseline_f1},
    {"modelo": "DistilBERT + LoRA", **{key: lora_metrics[key] for key in ["precision_macro", "recall_macro", "f1_macro"]}},
])
display(comparison)

              precision    recall  f1-score   support

    Business       0.83      0.82      0.83       500
    Sci_Tech       0.85      0.88      0.86       500
      Sports       0.96      0.98      0.97       500
       World       0.90      0.88      0.89       500

    accuracy                           0.89      2000
   macro avg       0.89      0.89      0.89      2000
weighted avg       0.89      0.89      0.89      2000

Matriz de confusion LoRA:
[[411  61   6  22]
 [ 40 438   1  21]
 [  6   3 488   3]
 [ 36  14  12 438]]


,modelo,precision_macro,recall_macro,f1_macro
0,TF-IDF + LogisticRegression,0.896151,0.8960,0.896003
1,DistilBERT + LoRA,0.887498,0.8875,0.887365


## 7. Guardado de evidencia

Guardar los resultados, la configuracion y el tiempo de entrenamiento para incorporarlos al reporte PDF.

In [8]:
evidence = {
    "model_name": MODEL_NAME,
    "lora": {
        "r": 8,
        "alpha": 16,
        "dropout": 0.1,
        "target_modules": ["q_lin", "v_lin"],
    },
    "total_parameters": total_parameters,
    "trainable_parameters": trainable_parameters,
    "trainable_percentage": trainable_percentage,
    "training_seconds": training_seconds,
    "final_train_loss": train_result.metrics["train_loss"],
    "validation_metrics": validation_metrics,
    "test_metrics": lora_metrics,
    "lora_classification_report": lora_report,
    "baseline_classification_report": baseline_report,
    "baseline_test_metrics": {
        "precision_macro": baseline_precision,
        "recall_macro": baseline_recall,
        "f1_macro": baseline_f1,
    },
    "comparison": comparison.to_dict(orient="records"),
}
with open(ROOT / "resultados_lora.json", "w", encoding="utf-8") as file:
    json.dump(evidence, file, indent=2)
print("Evidencia guardada en resultados_lora.json")

Evidencia guardada en resultados_lora.json


## 8. Conclusion tecnica

En esta ejecucion, el baseline TF-IDF + LogisticRegression (F1 macro ~0.896) supero levemente a DistilBERT + LoRA (F1 macro ~0.885) en el conjunto de test. Para una tarea de clasificacion de topicos con vocabulario muy distintivo entre clases como AG News, la representacion TF-IDF ya captura casi toda la senal discriminativa, por lo que la capacidad contextual adicional de un Transformer no se traduce en una mejora de metricas. Sin embargo, el enfoque LoRA demuestra su valor real en eficiencia: solo se entrena ~1.1% de los parametros del modelo (741 mil de 67.7 millones), lo que permite adaptar un Transformer preentrenado en unos pocos minutos de GPU sin necesidad de actualizar todos sus pesos. En este caso concreto, el costo computacional adicional del fine-tuning no se justifica frente al modelo clasico por la mejora de metricas obtenida; el valor de LoRA se justificaria mejor en tareas mas complejas (matices semanticos, lenguaje ambiguo, dominios donde el vocabulario superficial no alcanza), donde la capacidad contextual del Transformer si aportaria una ventaja medible.